# DATATHON 2026 — Pipeline

**Scope of this notebook (current revision):** Phase 0 (Environment Setup) + Phase 1 (Data Ingestion & Initial Cleaning).
Downstream phases (Audit, Integration, Reconciliation, Feature Engineering, Modeling) will be appended later without breaking existing cells.

**Contract:** This notebook consumes schema/loader/audit utilities from `helpers.py`. All I/O, dtypes, date parsing, and categorical promotion are centralized there, so every future phase sees the *same* typed frames.

**Determinism:** `RANDOM_SEED = 42` is applied at import time in `helpers.py`.

**References:** `PIPELINE_SPEC.md` (single source of truth), `schema_diagram.mermaid` (ER diagram).


## Phase 0 — Environment Setup

In [ ]:
# 0.1  Imports & environment fingerprint
from __future__ import annotations

import sys, platform, importlib, time
from pathlib import Path

import numpy as np
import pandas as pd

# Project helpers (schema + loaders + audit utilities)
import helpers as H
importlib.reload(H)  # idempotent reload when iterating

print(f"Python     : {sys.version.split()[0]} ({platform.system()} {platform.release()})")
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"DATA_DIR   : {H.DATA_DIR}")
print(f"OUTPUT_DIR : {H.OUTPUT_DIR}")
print(f"SEED       : {H.RANDOM_SEED}")


In [ ]:
# 0.2  Sanity-check that every expected CSV is physically present
missing = [t for t in H.ALL_TABLES if not (H.DATA_DIR / f"{t}.csv").exists()]
assert not missing, f"Missing CSVs: {missing}"
print(f"All {len(H.ALL_TABLES)} tables present:", H.ALL_TABLES)


## Phase 1 — Data Ingestion & Initial Cleaning

### 1.1 Typed Load
Every table is loaded via `helpers.load_csv_typed`, which applies:
- Explicit dtypes (int32/int16/int8/float64/string) for memory + correctness.
- Date parsing for all columns listed in `DATE_COLUMNS`.
- Post-load promotion of low-cardinality string columns to `category` dtype.

This guarantees reproducibility of the downstream join, audit, and feature pipelines.


In [ ]:
# 1.1  Load every table into a dict of DataFrames
t0 = time.time()
tables = H.load_all()
print(f"Loaded {len(tables)} tables in {time.time()-t0:.2f}s\n")

summary = pd.DataFrame(
    [(k, len(v), len(v.columns), round(v.memory_usage(deep=True).sum()/1e6, 2))
     for k, v in tables.items()],
    columns=["table", "rows", "cols", "mem_MB"]
).sort_values("rows", ascending=False).reset_index(drop=True)
summary


### 1.2 Per-column Profile
For every column in every table we record: dtype, null count, null %, unique count, and min/max (numeric & datetime only).


In [ ]:
# 1.2  Column-level profile across ALL tables
profile = pd.concat(
    [H.profile_table(df, name) for name, df in tables.items()],
    ignore_index=True,
)

profile_path = H.OUTPUT_DIR / "profile_report.csv"
profile.to_csv(profile_path, index=False)
print(f"Wrote {profile_path}  ({len(profile)} column records)")
profile.head(20)


### 1.3 Primary-Key Integrity
Each declared PK must be unique within its table. `order_items` has no natural PK in the source schema, so we
audit `(order_id, product_id)` as a *composite candidate key* and explicitly flag duplicates (Gap G1 in the spec).


In [ ]:
# 1.3  PK uniqueness checks
pk_specs = [
    ("customers",         ["customer_id"],               "PK_customers"),
    ("products",          ["product_id"],                "PK_products"),
    ("promotions",        ["promo_id"],                  "PK_promotions"),
    ("geography",         ["zip"],                       "PK_geography"),
    ("orders",            ["order_id"],                  "PK_orders"),
    ("payments",          ["order_id"],                  "PK_payments"),
    ("shipments",         ["order_id"],                  "PK_shipments"),
    ("returns",           ["return_id"],                 "PK_returns"),
    ("reviews",           ["review_id"],                 "PK_reviews"),
    ("order_items",       ["order_id", "product_id"],    "CAND_order_items_oid_pid"),  # G1 flag
    ("inventory",         ["product_id", "year", "month"], "PK_inventory_pm"),
    ("sales",             ["Date"],                      "PK_sales_date"),
    ("sample_submission", ["Date"],                      "PK_submission_date"),
    ("web_traffic",       ["date"],                      "PK_web_traffic_date"),
]
pk_results = pd.DataFrame(
    [H.audit_pk_unique(tables[t], keys, name) for t, keys, name in pk_specs]
)
pk_results


### 1.4 Foreign-Key Integrity
Every FK defined in the ER diagram is audited for orphan values (child-key values not found in the parent PK).


In [ ]:
# 1.4  FK orphan checks
fk_specs = [
    # child_table, child_key, parent_table, parent_key, label
    ("customers",   "zip",         "geography", "zip",         "FK_customers_zip"),
    ("orders",      "customer_id", "customers", "customer_id", "FK_orders_customer"),
    ("orders",      "zip",         "geography", "zip",         "FK_orders_zip"),
    ("order_items", "order_id",    "orders",    "order_id",    "FK_order_items_order"),
    ("order_items", "product_id",  "products",  "product_id",  "FK_order_items_product"),
    ("order_items", "promo_id",    "promotions","promo_id",    "FK_order_items_promo"),
    ("order_items", "promo_id_2",  "promotions","promo_id",    "FK_order_items_promo2"),
    ("payments",    "order_id",    "orders",    "order_id",    "FK_payments_order"),
    ("shipments",   "order_id",    "orders",    "order_id",    "FK_shipments_order"),
    ("returns",     "order_id",    "orders",    "order_id",    "FK_returns_order"),
    ("returns",     "product_id",  "products",  "product_id",  "FK_returns_product"),
    ("reviews",     "order_id",    "orders",    "order_id",    "FK_reviews_order"),
    ("reviews",     "product_id",  "products",  "product_id",  "FK_reviews_product"),
    ("reviews",     "customer_id", "customers", "customer_id", "FK_reviews_customer"),
    ("inventory",   "product_id",  "products",  "product_id",  "FK_inventory_product"),
]
fk_results = pd.DataFrame(
    [H.audit_fk(tables[ct], ck, tables[pt], pk, name) for ct, ck, pt, pk, name in fk_specs]
)
fk_results


### 1.5 Consolidated Audit Export

In [ ]:
# 1.5  Merge PK + FK results into a single audit_results.csv
audit = pd.concat([
    pk_results.assign(kind="PK"),
    fk_results.assign(kind="FK"),
], ignore_index=True)

audit_path = H.OUTPUT_DIR / "audit_results.csv"
audit.to_csv(audit_path, index=False)
print(f"Wrote {audit_path}  ({len(audit)} checks)")

n_fail = int((audit["verdict"] == "FAIL").sum())
print(f"FAIL count: {n_fail}")
audit[["kind", "check_id", "verdict"]]


### 1.6 Immediate Anomaly Flags (empirical findings from Phase 1a/1b probes)

These are anomalies discovered while exercising the real data against the spec. They are *logged*, not *fixed* — corrections happen in Phase 2 (Audit) after user review.

| # | Finding | Impact / Resolution |
|---|---------|---------------------|
| A1 | `order_items` has **16 duplicate** `(order_id, product_id)` pairs | Confirms **Gap G1** — introduce surrogate `line_item_id` before any groupby on that key. |
| A2 | `orders.order_status` has **6 distinct values** (`delivered`, `cancelled`, `returned`, `shipped`, `paid`, `created`) rather than the 4 referenced in spec §3 | Treat all six as legitimate; revenue reconciliation uses the full line-item universe anyway (see A5). |
| A3 | `web_traffic` starts **2013-01-01**, but `sales.csv` starts **2012-07-04** | New **Gap G13** — 6-month web-traffic gap at the head of the training period. Feature must tolerate NaNs or impute. |
| A4 | `promo_id_2` is populated on **206** rows where the referenced promo's `stackable_flag = 0` | Original validation rule B7 (*"stacked promos must both be stackable"*) is unenforceable as written. Recommendation: *drop B7* and treat `promo_id_2` as a second-promo field with no stackability constraint. |
| A5 | `sales.Revenue` equals `Σ(quantity × unit_price)` over **all** order_items (no status filter, no refund subtraction, no discount subtraction) with MAE = 0.00 | Revenue formula is canonical **F1**. This pre-answers Phase 5 reconciliation. `discount_amount`, shipping, refunds are **not** part of `Revenue` as defined by the scoring table. |
| A6 | `order_status ∈ {cancelled, returned}` coexists with non-null `payments`/`shipments` rows for some orders | Does **not** violate Revenue (because F1 includes them). Flag for Phase 2 logical-consistency report only. |
| A7 | Normalization violations **N1** (payments.payment_method vs orders.payment_method), **N2** (customers.city vs geography.city), **N3** (inventory denormalized) all show **zero divergence** | Pure redundancy. Safe to drop redundant columns at join time per decisions **D5/D6**. |
| A8 | `web_traffic` cardinality = **1 row per date** (source column = single channel per day) | **Gap G6 resolved.** No aggregation needed. |
| A9 | `sample_submission` schema = `(Date, Revenue, COGS)` with 548 rows | **Gap G9 resolved.** Submission horizon matches spec. |
| A10 | `sales.csv` filename (not `sales_train.csv`) with 3833 rows covering 2012-07-04 → 2022-12-31 | **Gap G10 resolved.** No renaming ambiguity. |


In [ ]:
# 1.6  Persist a lightweight snapshot for downstream phases (avoids re-reading CSVs)
snapshot_dir = H.OUTPUT_DIR / "parquet"
snapshot_dir.mkdir(exist_ok=True)
for name, df in tables.items():
    df.to_parquet(snapshot_dir / f"{name}.parquet", index=False)
print(f"Snapshotted {len(tables)} tables to {snapshot_dir}")


## Summary — Phase 0 & Phase 1 Gate

- **Environment reproducible.** Seed, dtypes, and date parsing fixed in `helpers.py`.
- **All 14 CSVs load successfully** with typed schemas.
- **Primary keys**: 10/10 table-level PKs PASS. `order_items(order_id, product_id)` shows 16 duplicates — expected (Gap G1).
- **Foreign keys**: 15/15 PASS (zero orphans).
- **Profile & audit artifacts** exported to `outputs/profile_report.csv` and `outputs/audit_results.csv`.
- **Parquet snapshot** written to `outputs/parquet/` for fast re-entry into Phase 2+.

**Ready to proceed to Phase 2 — Multi-layer Audit** upon user approval.


## Phase 2 — Multi-Layer Audit

Runs:
- **§4.3** Business rules B1–B15
- **§4.7** Granularity alignment G_CHK1–G_CHK5
- **G1** Generates `line_item_id` surrogate key on `order_items`

Outputs:
- `outputs/phase2_audit_results.csv` (22 rule checks)
- `outputs/parquet/order_items.parquet` (updated — now carries `line_item_id`)
- `outputs/phase2_summary.md` (human-readable triage)

See `outputs/phase2_summary.md` for the narrative triage of all FAIL / LOG / WARN verdicts.


In [ ]:
# Phase 2 — execute via the standalone script to keep notebook lean
import subprocess, sys
from pathlib import Path

p = Path("outputs/phase2_audit.py")
if not p.exists():
    raise FileNotFoundError(p.resolve())
r = subprocess.run([sys.executable, str(p)], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(r.stderr)


In [ ]:
# Load the audit results and render the verdict matrix
import pandas as pd
phase2 = pd.read_csv("outputs/phase2_audit_results.csv")
phase2["verdict"] = pd.Categorical(phase2["verdict"], ["PASS","LOG","WARN","FAIL"])
phase2.sort_values(["verdict","rule_id"])[["rule_id","metric","threshold","verdict","detail"]]


In [ ]:
# Confirm the line_item_id surrogate is in place on the Phase 2 snapshot
oi2 = pd.read_parquet("outputs/parquet/order_items.parquet")
assert "line_item_id" in oi2.columns
assert oi2["line_item_id"].is_unique
print(f"order_items rows: {len(oi2):,}   line_item_id unique: {oi2['line_item_id'].is_unique}")
oi2.head(3)


### Phase 2 exit gate

- PK violations: **0** (composite `(order_id, product_id)` duplicates resolved via `line_item_id` surrogate).
- FK orphan rate: **0.0%** across 15 relationships (Phase 1).
- Business-rule failures: surfaced 3 FAILs, all triaged in `phase2_summary.md`:
  - **B10_has_ship (564)** & **B11 (80)** — dataset-tail truncation (Dec 2022); keep rows, flag missing shipments/returns.
  - **B13 (80,623)** — `signup_date` is not "first interaction"; use `min(order_date)` as the tenure anchor.
- Decisions folded into the spec: B7 deprecated; `line_item_id` is the new order_items PK; signup_date redefined.

**Ready to proceed to Phase 3 — Pre-join aggregation of satellite tables.**
